# সহজ ভাষায় Notebook Guide

এই notebook-এ project-এর **একই `app/` code** (যেটা `uvicorn app.main:app` চালায়) cell-by-cell খুলে দেখা হয়েছে — RAG core থেকে শুরু করে HTTP layer, streaming, backpressure আর degradation পর্যন্ত। Technical term English-এ, explanation Bangla-তে।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে run করুন। Kernel = project-এর `venv`।
2. Section 4-7 real Ollama (`llama3.1:8b`) লাগে: `ollama serve`। বাকিগুলো Ollama ছাড়াই চলে।
3. প্রতিটা সংখ্যা (latency, distance) আপনার মেশিনে আলাদা হবে — নিজে মেপে নিজের সিদ্ধান্ত নিন।

# Production-Ready RAG API — Exploration

1. PDF → pages → chunks (আর কেন chunk কখনো page পার হয় না)
2. Guardrail calibration — `max_distance` কোথা থেকে আসে
3. Idempotent ingestion — content-hash chunk id
4. HTTP layer in-process: upload → ask → latency breakdown
5. Streaming: TTFT বনাম total time
6. Answer cache + corpus version
7. Graceful degradation: LLM down হলে কী হয়
8. Backpressure: semaphore → 503 + `Retry-After`

In [1]:
import sys, os, tempfile, time, json, asyncio
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Notebook-এর নিজের temp index — project-এর data/chroma_db ছোঁয় না
os.environ["RAG_CHROMA_DIR"] = tempfile.mkdtemp(prefix="rag_nb_")

import pandas as pd
from app.settings import get_settings
from app.rag.ingest import extract_pages, chunk_pages
from app.rag.components import Embedder, ChromaStore
from app.rag.engine import RagEngine, build_engine

s = get_settings()
PDF = (project_root / "samples" / "employee_handbook.pdf").read_bytes()
pd.Series({k: getattr(s, k) for k in ["model", "embedding_model", "chunk_size", "chunk_overlap", "top_k",
                                      "max_distance", "max_concurrent_llm", "queue_timeout_s", "cache_size"]}, name="value").to_frame()

,value
model,llama3.1:8b
embedding_model,sentence-transformers/paraphrase-MiniLM-L3-v2
chunk_size,300
chunk_overlap,60
top_k,4
max_distance,26.0
max_concurrent_llm,1
queue_timeout_s,30.0
cache_size,256


## 1. PDF → Pages → Chunks

`samples/employee_handbook.pdf` একটা ছোট কাল্পনিক ৩-page policy (`python samples/make_handbook.py` বানায়)। Extraction-এর আগে তিনটা trust-boundary check: `%PDF-` magic bytes, খোলা যায় কিনা, page limit। Text layer না থাকলে (scanned PDF) পরিষ্কার error — OCR scope-এর বাইরে।

Chunker: আগে paragraph, লম্বা হলে sentence, একদম শেষে hard cut। **Overlap শুধু একই page-এর ভেতরে** — chunk কখনো page পার হয় না, তাই citation-এর page number সবসময় সঠিক।

In [2]:
pages = extract_pages(PDF, "employee_handbook.pdf")
chunks = chunk_pages(pages, "employee_handbook.pdf", s.chunk_size, s.chunk_overlap)
print(len(pages), "pages ->", len(chunks), "chunks")
pd.DataFrame([{"id": c.id, "page": c.page, "chars": len(c.text), "starts": c.text[:60].replace("\n", " ")} for c in chunks])

3 pages -> 7 chunks


,id,page,chars,starts
0,cc0862fa120969b5,1,287,Acme Corp Employee Handbook - Leave Policy Ann...
1,eb865f3100171db4,1,238,to the next year; anything above 5 days expire...
2,a09f65d4973efc24,1,246,ote is required for absences longer than 3 con...
3,2e3471ce5f492888,2,272,Acme Corp Employee Handbook - IT Security Poli...
4,519ce56b9e761441,2,246,portal. Every company laptop must use full-di...
5,5926526b3078daff,3,289,Acme Corp Employee Handbook - Remote Work and ...
6,a4743ca2998b8add,3,210,"office allowance of 300 USD covers a desk, cha..."


In [3]:
from app.rag.ingest import BadDocument
for name, data in [("notes.pdf", b"hello"), ("broken.pdf", b"%PDF-1.4 garbage"), ("handbook.pdf", PDF)]:
    try:
        extract_pages(data, name, max_pages=2)
    except BadDocument as e:
        print(f"{name:13} -> BadDocument: {e}")

notes.pdf     -> BadDocument: 'notes.pdf' is not a PDF (missing %PDF- header)
broken.pdf    -> BadDocument: Could not open 'broken.pdf' as a PDF: Stream has ended unexpectedly
handbook.pdf  -> BadDocument: 'handbook.pdf' has 3 pages; the limit is 2


## 2. Guardrail Calibration

Retrieval সবসময় *কিছু একটা* ফেরত দেয় — এমনকি "France-এর রাজধানী কী?" প্রশ্নেও। তাই সবচেয়ে কাছের chunk-এর L2 distance একটা threshold-এর বেশি হলে LLM-কে ডাকাই হয় না, সরাসরি "not found"। এতে (১) hallucination কমে, (২) off-topic প্রশ্নে GPU সময় শূন্য।

Threshold কোনো magic number না — **নিজের document আর প্রশ্নে মাপতে হয়।** নিচে ৪টা chunk size-এ on-topic প্রশ্নের *সবচেয়ে খারাপ* distance আর off-topic প্রশ্নের *সবচেয়ে ভালো* distance। দুটোর মাঝে ফাঁক থাকলেই threshold বসানো যায়।

In [4]:
emb = Embedder(s.embedding_model)
ON = ["How many sick days do employees get?", "How many characters must a password have?",
      "How many days per week can I work remotely?", "How much is the home-office allowance?",
      "What is the meal limit when travelling?", "How long is parental leave?", "Can I carry over unused leave?"]
OFF = ["What is the capital of France?", "Who won the 2018 World Cup?", "How do I bake bread?", "What is Python's GIL?"]

rows = []
for size, ov in [(700, 100), (400, 80), (300, 60), (200, 40)]:
    e = RagEngine(emb, ChromaStore(tempfile.mkdtemp(), "calib"), None, chunk_size=size, chunk_overlap=ov)
    n = e.ingest("h.pdf", PDF).chunks
    on = [e.retrieve(q)[0][0].distance for q in ON]
    off = [e.retrieve(q)[0][0].distance for q in OFF]
    rows.append({"chunk_size": size, "chunks": n, "on_topic_max": round(max(on), 1),
                 "off_topic_min": round(min(off), 1), "gap": round(min(off) - max(on), 1)})
calib = pd.DataFrame(rows); calib

/Users/salmansakib/Documents/Projects/interview-Prep-Case-Study/Job Ready AI Bootcamp/week 15/Class 1 Project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1025 20:37:31.624000 65677 venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 10763.16it/s]

,chunk_size,chunks,on_topic_max,off_topic_min,gap
0,700,3,25.8,25.6,-0.2
1,400,6,25.0,25.9,0.8
2,300,7,25.2,27.2,2.0
3,200,12,23.5,27.3,3.8


**পড়ার নিয়ম:** `gap` ঋণাত্মক মানে কোনো threshold-ই দুই দলকে আলাদা করতে পারে না — কিছু সঠিক প্রশ্ন বাদ যাবে অথবা কিছু ভুল প্রশ্ন ঢুকবে। বড় chunk (পুরো page) অনেক topic মিশিয়ে embedding-কে "ঘোলা" করে; ছোট chunk-এ embedding তীক্ষ্ণ। কিন্তু খুব ছোট chunk-এ LLM-এর context কম পড়ে। এই project `300/60` আর `max_distance = 26.0` বেছেছে — মাঝখানে, দুই দিকেই margin।

⚠️ এটা মাত্র ১১টা প্রশ্ন। Production-এ ৫০-১০০টা labelled প্রশ্নে মাপুন, আর document বদলালে আবার মাপুন।

## 3. Idempotent Ingestion

Chunk id = `sha256(source + "\0" + text)[:16]`। একই file আবার upload → একই id → Chroma `upsert` নতুন কিছু যোগ করে না। Client retry করলে (network timeout-এর পর খুব সাধারণ) duplicate chunk হয় না — duplicate থাকলে top-k একই passage দিয়ে ভরে যেত।

In [5]:
engine = build_engine(s)
first, again = engine.ingest("employee_handbook.pdf", PDF), engine.ingest("employee_handbook.pdf", PDF)
print("first :", first)
print("again :", again)
print("stored:", engine.store.count(), "chunks")

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 19815.04it/s]

first : IngestResult(filename='employee_handbook.pdf', pages=3, chunks=7, new_chunks=7)
again : IngestResult(filename='employee_handbook.pdf', pages=3, chunks=7, new_chunks=0)
stored: 7 chunks


## 4. HTTP Layer In-Process

`TestClient` আসল ASGI app চালায় — middleware, validation, exception handler, lifespan সব — শুধু network socket ছাড়া। `with TestClient(app)` lifespan চালায়; প্রথম request-এ service (embedder + Chroma + Ollama client) একবার তৈরি হয়।

Response-এর `timings_ms` আর `usage` দেখায় সময় কোথায় যায়: embed আর vector search কয়েক millisecond, **LLM কয়েক সেকেন্ড**। তাই optimization মানে আগে LLM-এর token কমানো (top-k, chunk size, `num_predict`) — Python code না।

In [6]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
client.__enter__()                       # run the lifespan (a `with` block in normal code)
print(client.get("/ready").json())
r = client.post("/v1/documents", files={"file": ("employee_handbook.pdf", PDF, "application/pdf")})
print(r.status_code, r.json())

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 16763.80it/s]

{'status': 'ready', 'backend': 'real', 'llm': 'ok', 'indexed_documents': 1}
201 {'filename': 'employee_handbook.pdf', 'pages': 3, 'chunks': 7, 'new_chunks': 0, 'request_id': '3186e1da8c7a'}


In [7]:
t0 = time.perf_counter()
r = client.post("/v1/ask", json={"question": "How many sick days do employees get, and when must I tell my manager?"})
wall = (time.perf_counter() - t0) * 1000
body = r.json()
print(body["answer"])
print("request id:", r.headers["x-request-id"], "| server time:", r.headers["x-process-time-ms"], "ms | client wall:", round(wall), "ms")
pd.DataFrame(body["citations"])[["n", "source", "page", "distance", "snippet"]]

Employees get 12 paid sick days per year. You must tell your manager before 10am on the first day of your absence. [1]
request id: 0b7dc7921e08 | server time: 3255.5 ms | client wall: 3257 ms


,n,source,page,distance,snippet
0,1,employee_handbook.pdf,1,8.43,to the next year; anything\nabove 5 days expir...
1,2,employee_handbook.pdf,3,16.15,Acme Corp Employee Handbook - Remote Work and ...
2,3,employee_handbook.pdf,1,19.20,Acme Corp Employee Handbook - Leave Policy\nAn...
3,4,employee_handbook.pdf,3,20.17,"office allowance of 300 USD covers a desk, cha..."


In [8]:
u = body["usage"]
breakdown = pd.DataFrame([
    {"stage": "embed question", "ms": body["timings_ms"]["embed_ms"]},
    {"stage": "vector search", "ms": body["timings_ms"]["search_ms"]},
    {"stage": "LLM (prefill + decode)", "ms": body["timings_ms"]["llm_ms"]},
])
breakdown["share_%"] = (100 * breakdown.ms / breakdown.ms.sum()).round(1)
print(f"prompt tokens {u['prompt_tokens']}, completion tokens {u['completion_tokens']}, decode speed {u['tokens_per_s']} tok/s, model load {u['load_ms']} ms")
breakdown

prompt tokens 402, completion tokens 31, decode speed 18.9 tok/s, model load 2.4 ms


,stage,ms,share_%
0,embed question,5.5,0.2
1,vector search,0.6,0.0
2,LLM (prefill + decode),3248.4,99.8


**Prefill বনাম decode:** prompt-এর সব token একসাথে পড়া (prefill) TTFT ঠিক করে; তারপর প্রতিটা নতুন token আলাদা করে লেখা (decode) memory bandwidth-এ আটকে থাকে, তাই tok/s প্রায় স্থির। মোট সময় ≈ prefill + completion_tokens ÷ tok/s। Context দ্বিগুণ হলে prefill বাড়ে; উত্তর দ্বিগুণ লম্বা হলে decode দ্বিগুণ হয়।

Validation আর structured error — প্রতিটা error-এর একই আকার (`error`, `detail`, `request_id`), যাতে client code একটাই parser লেখে:

In [9]:
for desc, call in [
    ("empty question", lambda: client.post("/v1/ask", json={"question": ""})),
    ("top_k too big", lambda: client.post("/v1/ask", json={"question": "q", "top_k": 50})),
    (".txt upload", lambda: client.post("/v1/documents", files={"file": ("a.txt", b"x", "text/plain")})),
    ("fake .pdf", lambda: client.post("/v1/documents", files={"file": ("a.pdf", b"hello", "application/pdf")})),
]:
    r = call(); print(f"{desc:15} {r.status_code}  {r.json()}")

empty question  422  {'error': 'validation_error', 'detail': 'body.question: String should have at least 1 character', 'request_id': '7e48d8d5b37f'}
top_k too big   422  {'error': 'validation_error', 'detail': 'body.top_k: Input should be less than or equal to 10', 'request_id': '8d7989aa6573'}
.txt upload     415  {'error': 'http_error', 'detail': 'Only .pdf files are supported', 'request_id': '19a7e865e631'}
fake .pdf       422  {'error': 'bad_document', 'detail': "'a.pdf' is not a PDF (missing %PDF- header)", 'request_id': 'c8b2bfd9ab2c'}


## 5. Streaming: TTFT বনাম Total

`/v1/ask/stream` Server-Sent Events পাঠায়: `citations` (একবার) → `token` (অনেকবার) → `done`। ব্যবহারকারী প্রথম শব্দ দেখে **TTFT**-এ, পুরো উত্তর পায় total-এ। Streaming মোট সময় কমায় না — অপেক্ষার *অনুভূতি* কমায়। তাই load test-এ দুটো আলাদা মাপতে হয়।

In [10]:
t0 = time.perf_counter(); ttft = None; events = []
with client.stream("POST", "/v1/ask/stream", json={"question": "How many characters must a password have?"}) as resp:
    print(resp.headers["content-type"], "| X-Accel-Buffering:", resp.headers["x-accel-buffering"])
    for line in resp.iter_lines():
        if line.startswith("event: "):
            events.append(line[7:])
            if line == "event: token" and ttft is None:
                ttft = time.perf_counter() - t0
        elif line.startswith("data: ") and events[-1] == "done":
            done = json.loads(line[6:])
total = time.perf_counter() - t0
print(f"TTFT {ttft:.2f}s | total {total:.2f}s | events: {events[0]}, {events.count('token')} x token, {events[-1]}")
print("answer:", done["answer"])

text/event-stream; charset=utf-8 | X-Accel-Buffering: no


TTFT 0.33s | total 1.15s | events: citations, 14 x token, done
answer: According to [1], passwords must have at least 14 characters.


### Prompt-prefix cache: TTFT কেন কখনো ০.১ s, কখনো ২ s?

উপরের TTFT ছোট হতে পারে কারণ Ollama আগের prompt-এর **KV cache** রেখে দেয়: নতুন prompt-এর শুরুটা (prefix) আগেরটার সাথে মিললে সেই অংশের prefill আর করতে হয় না। তাই prompt-এ **স্থির অংশ আগে** (system prompt, passage), **প্রশ্ন সবার শেষে**। নিচে একটা nonce দিয়ে cache ইচ্ছে করে ভাঙা হয়েছে, তারপর চারটা অবস্থা মাপা:

In [11]:
from app.rag.engine import SYSTEM_PROMPT

def measure(prompt):
    for kind, v in engine.llm.stream(SYSTEM_PROMPT, prompt):
        if kind == "usage":
            return {"prompt_tokens": v.prompt_tokens, "prefill_ms": round(v.prefill_ms), "ttft_ms": round(v.ttft_ms)}

hits, _ = engine.retrieve("How many sick days do employees get?")
nonce = f"(session {time.time_ns()})\n"           # makes the passages a never-seen prefix
p1 = nonce + RagEngine.prompt("How many sick days do employees get?", hits)
pd.DataFrame([
    {"case": "1. new prompt (cold prefix)", **measure(p1)},
    {"case": "2. exact same prompt again", **measure(p1)},
    {"case": "3. same passages, NEW question at the end", **measure(nonce + RagEngine.prompt("When must I tell my manager?", hits))},
    {"case": "4. new question at the START", **measure(f"Q: is a doctor's note needed? ({time.time_ns()})\n" + p1)},
])

,case,prompt_tokens,prefill_ms,ttft_ms
0,1. new prompt (cold prefix),404,2255,2271
1,2. exact same prompt again,404,50,64
2,"3. same passages, NEW question at the end",403,138,156
3,4. new question at the START,422,2101,2118


**পড়ার নিয়ম:** (১) পুরো prefill — কয়েকশো token, প্রায় ২ s। (২) আর (৩) প্রায় সব token cache থেকে — কয়েক ms। (৪) প্রশ্ন শুরুতে থাকলে পরের সব token-এর prefix বদলে যায় → আবার পুরো prefill। Monitoring-এ `prefill_ms` আর TTFT-এর এই দুই-চূড়ার (bimodal) বণ্টন দেখা যায় — load test-এ সব প্রশ্ন একই হলে latency মিথ্যা রকম ভালো দেখায়।

## 6. Answer Cache + Corpus Version

Exact-match cache: key = (normalized question, source, top_k, **corpus version**)। নতুন document এলে version বাড়ে, পুরোনো উত্তর নিজে থেকেই অকেজো — কারণ নতুন document উত্তর বদলে দিতে পারে। Semantic ("কাছাকাছি প্রশ্ন") cache ইচ্ছে করে নেই: ভুল hit একটা আত্মবিশ্বাসী ভুল উত্তর দেয়, আর গবেষণায় দেখা গেছে এটা poison-ও করা যায়।

In [12]:
def timed(q):
    t0 = time.perf_counter(); b = client.post("/v1/ask", json={"question": q}).json()
    return {"question": q, "cached": b["cached"], "client_ms": round((time.perf_counter() - t0) * 1000, 1)}

rows = [timed("How long is parental leave?"), timed("  how long is PARENTAL leave? ")]
from samples.make_handbook import build      # a second, different document bumps the corpus version
extra = Path(tempfile.mkdtemp()) / "handbook_v2.pdf"; build(extra)
client.post("/v1/documents", files={"file": ("handbook_v2.pdf", extra.read_bytes(), "application/pdf")})
rows.append(timed("How long is parental leave?"))
pd.DataFrame(rows)

,question,cached,client_ms
0,How long is parental leave?,False,1424.1
1,how long is PARENTAL leave?,True,3.6
2,How long is parental leave?,False,2965.0


`handbook_v2.pdf`-এর text একই, কিন্তু source আলাদা, তাই chunk id আলাদা → নতুন chunk → version বাড়ে → তৃতীয় call cache miss।

## 7. Graceful Degradation

Ollama বন্ধ বা timeout হলে 500 error দেওয়ার বদলে service retrieval-only উত্তর দেয়: `degraded: true` + সবচেয়ে প্রাসঙ্গিক passage। ব্যবহারকারী অন্তত source পায়, আর monitoring `degraded` গুনে alert দিতে পারে। Degraded উত্তর cache হয় না।

In [13]:
from app.rag.components import OllamaLLM
svc = app.state.service
real_llm = svc.backend.engine.llm
svc.backend.engine.llm = OllamaLLM("http://localhost:11999", s.model, timeout_s=3)    # nothing listens there
try:
    b = client.post("/v1/ask", json={"question": "What is the meal limit when travelling?"}).json()
    print("degraded:", b["degraded"], "| found:", b["found"], "| citations:", len(b["citations"]))
    print(b["answer"]); print("top passage:", b["citations"][0]["snippet"][:120])
    print("/ready ->", client.get("/ready").status_code)
finally:
    svc.backend.engine.llm = real_llm
print("/ready after restore ->", client.get("/ready").status_code)

degraded: True | found: True | citations: 4
The language model is unavailable right now. These are the most relevant passages from your documents.
top passage: office allowance of 300 USD covers a desk, chair or monitor.

Receipts must be submitted
within 30 days of purchase.

Tr
/ready -> 503
/ready after restore -> 200


## 8. Backpressure: Semaphore → 503 + Retry-After

Ollama default-এ model-প্রতি একটাই request একসাথে চালায় (`OLLAMA_NUM_PARALLEL=1`), বাকিগুলো নিজের ভেতরে queue করে। আমাদের `max_concurrent_llm` সেটার সমান রাখা হয়েছে, যাতে queue **আমাদের** layer-এ থাকে — সেখানে timeout দেওয়া যায়, মাপা যায়। Slot `queue_timeout_s`-এর মধ্যে না পেলে দ্রুত 503 + `Retry-After` — অসীম অপেক্ষার চেয়ে সৎ ব্যর্থতা ভালো; client বা load balancer অন্য replica-য় যায়।

এখানে `FakeBackend` (প্রতি উত্তর ০.৬s) — Ollama ছাড়াই আচরণটা দেখা যায়। `ASGITransport` lifespan চালায় না, তাই settings হাতে বসানো।

In [14]:
import httpx
from app.backend import FakeBackend
from app.main import get_service
from app.service import RagService

slow = RagService(FakeBackend(delay=0.6), max_concurrent=1, queue_timeout_s=1.0, cache_size=0)
slow.backend.docs["employee_handbook.pdf"] = "x"
app.dependency_overrides[get_service] = lambda: slow

async def burst(n):
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://nb") as ac:
        async def one(i):
            t0 = time.perf_counter(); r = await ac.post("/v1/ask", json={"question": f"q{i}"})
            return {"req": i, "status": r.status_code, "seconds": round(time.perf_counter() - t0, 2),
                    "retry_after": r.headers.get("retry-after")}
        return await asyncio.gather(*(one(i) for i in range(n)))

try:
    out = pd.DataFrame(await burst(5))
finally:
    app.dependency_overrides.clear()
out

,req,status,seconds,retry_after
0,0,200,0.61,NaN
1,1,200,1.22,NaN
2,2,503,1.01,1
3,3,503,1.01,1
4,4,503,1.01,1


**পড়ার নিয়ম:** ১ম request সাথে সাথে চলে (০.৬s), ২য়টা slot পায় ১ম শেষ হলে (≈১.২s)। বাকিরা ১s অপেক্ষা করে slot পায় না → 503, আর এটা ঘটে ~১s-এ, ১০s পরে না। Benchmark (`python benchmark/bench.py --fake --queue-timeout 0.5`) একই জিনিস concurrency sweep-এ দেখায়।

In [15]:
client.__exit__(None, None, None)
print("closed")

closed


## 📝 Key Takeaways

* **সময় যায় LLM-এ।** Embed + search কয়েক ms, LLM কয়েক সেকেন্ড। Latency কমানোর lever: context token (top-k × chunk size), `num_predict`, model warm রাখা (`keep_alive`)।
* **Guardrail threshold মাপতে হয়।** Chunk size বদলালে distance-এর বণ্টন বদলায় — threshold আবার calibrate করুন।
* **Idempotency hash দিয়ে।** Content-hash id + `upsert` → retry নিরাপদ।
* **Backpressure গোপন queue-র চেয়ে ভালো।** Gate = Ollama-র parallel slot; বাকিদের দ্রুত 503 + `Retry-After`।
* **Degrade, don't die.** LLM না থাকলে passage ফেরত দিন, `degraded` flag দিয়ে।
* **TTFT আর total আলাদা মেট্রিক।** Streaming প্রথমটা কমায়, দ্বিতীয়টা না।

## 🧠 Brain Teasers

1. `RAG_MAX_CONCURRENT_LLM=2` দিয়ে server চালিয়ে `bench.py --stream --levels 2` চালান। TTFT কেন বাড়ে, কিন্তু throughput বাড়ে না?
2. Section 2-এ ৫টা নিজের on-topic আর ৫টা off-topic প্রশ্ন যোগ করুন। `max_distance = 26` কি এখনো টেকে?
3. Cache key থেকে `corpus_version` বাদ দিলে কোন ভুল দেখা দেবে? একটা test লিখে দেখান।
4. Upload-এর সময় শুধু `.pdf` extension দেখলে কী আক্রমণ সম্ভব? `%PDF-` magic check কী আটকায়, কী আটকায় না?